# Health Eat — 학습 → 예측 → Kaggle 제출

**실행 순서**: 위에서 아래로 셀을 순서대로 실행하세요.  
**런타임**: GPU (T4 이상) 권장 — 런타임 → 런타임 유형 변경 → T4 GPU

| 단계 | 내용 |
|---|---|
| 1 | 환경 설정 (repo clone + 패키지 설치) |
| 2 | Kaggle 인증 (Colab Secrets) |
| 3 | 데이터 다운로드 |
| 4 | 학습 (`train.py`) |
| 5 | 예측 + 제출 파일 생성 (`predict.py`) |
| 6 | Kaggle 제출 |

## 1. 환경 설정

In [ ]:
import os

REPO_URL = "https://github.com/beomjinkim2000/Code_IT_Team_1_FirstProject.git"
PROJECT_DIR = "/content/Code_IT_Team_1_FirstProject"

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    !git -C {PROJECT_DIR} pull

%cd {PROJECT_DIR}

In [ ]:
!pip install -q \
    ultralytics \
    torchmetrics[detection] \
    albumentations \
    kagglehub \
    kaggle \
    pyyaml \
    tqdm

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Kaggle 인증

**최초 1회 설정** (이후 재실행 시 자동 적용)

1. 왼쪽 사이드바 **🔑 아이콘** 클릭
2. **`KAGGLE_USERNAME`** → Kaggle 닉네임 입력 후 저장
3. **`KAGGLE_KEY`** → [Kaggle 계정](https://www.kaggle.com/settings/account) → API → **Create New Token** → kaggle.json 열어서 `key` 값 복사 후 저장
4. 각 Secret의 **노트북 액세스 허용** 토글 ON

In [ ]:
import json
import os
from google.colab import userdata

kaggle_username = userdata.get("KAGGLE_USERNAME")
kaggle_key      = userdata.get("KAGGLE_KEY")

os.environ["KAGGLE_USERNAME"] = kaggle_username
os.environ["KAGGLE_KEY"]      = kaggle_key

kaggle_dir = os.path.expanduser("~/.kaggle")
os.makedirs(kaggle_dir, exist_ok=True)
with open(f"{kaggle_dir}/kaggle.json", "w") as f:
    json.dump({"username": kaggle_username, "key": kaggle_key}, f)
os.chmod(f"{kaggle_dir}/kaggle.json", 0o600)

print(f"Kaggle 인증 완료 — 사용자: {kaggle_username}")

## 3. 데이터 다운로드

In [ ]:
from pathlib import Path
import kagglehub

COMPETITION_ID = "ai11-level1-project"

raw_data_path = Path("data/raw/ai11-level1-project")
dataset_path  = raw_data_path / "sprint_ai_project1_data"
required_dirs = [
    dataset_path / "train_images",
    dataset_path / "test_images",
    dataset_path / "train_annotations",
]

if all(d.exists() for d in required_dirs):
    print("데이터 이미 존재 — 다운로드 건너뜀")
else:
    raw_data_path.mkdir(parents=True, exist_ok=True)
    download_path = kagglehub.competition_download(
        COMPETITION_ID,
        output_dir=str(raw_data_path),
    )
    print(f"다운로드 완료: {download_path}")

print("\n데이터 구조:")
for d in required_dirs:
    count = len(list(d.glob("*"))) if d.exists() else 0
    print(f"  {d.name}: {count}개")

## 4. 학습

`configs/default.yaml`에서 설정 조정:
- `train.epochs`: 에폭 수 (기본 50)
- `train.batch_size`: 배치 크기 (기본 16, GPU 메모리 부족 시 8로 낮추기)
- `train.lr`: 학습률 (기본 0.001)

In [ ]:
!cat configs/default.yaml

In [ ]:
!python train.py --config configs/default.yaml

In [ ]:
!ls -lh outputs/checkpoints/

## 4-2. 모델 Drive 백업 (선택)

`best_model.pt`를 Google Drive에 저장합니다. Colab 세션이 끊겨도 모델이 보존됩니다.

In [ ]:
# (선택) Google Drive에 best_model.pt 백업
from google.colab import drive
import shutil
from pathlib import Path

drive.mount('/content/drive')

drive_dir = Path('/content/drive/MyDrive/HealthEat/checkpoints')
drive_dir.mkdir(parents=True, exist_ok=True)

src = Path('outputs/checkpoints/best_model.pt')
if src.exists():
    shutil.copy(src, drive_dir / 'best_model.pt')
    print(f'Drive 저장 완료: {drive_dir}/best_model.pt')
else:
    print('best_model.pt 없음 — 학습이 완료되었는지 확인하세요')

## 5. 예측 + 제출 파일 생성

`best_model.pt` 로드 → test 이미지 전체 예측 → `submission_v1.csv` + `predictions.json`

In [ ]:
!python predict.py --config configs/default.yaml

In [ ]:
import pandas as pd
df = pd.read_csv("outputs/submissions/submission_v1.csv")
print(f"행 수: {len(df)}")
print(f"이미지 수: {df['image_id'].nunique()}")
df.head(10)

## 6. Kaggle 제출

In [ ]:
SUBMIT_MESSAGE = "v1 - YOLOv8n baseline"

!kaggle competitions submit \
    -c {COMPETITION_ID} \
    -f outputs/submissions/submission_v1.csv \
    -m "{SUBMIT_MESSAGE}"

print("제출 완료!")

In [ ]:
!kaggle competitions submissions -c {COMPETITION_ID}